In [ ]:
import os
import csv
import time
import ccxt
import logging
import traceback
import numpy as np
import pandas as pd
from datetime import datetime
from binance.client import Client

from config import *

# ====================================================
# 1. 설정 및 상수 (Configuration)
# ====================================================

# [Mode Setting]
IS_LIVE_TRADING = False  # True: 실제 주문 전송, False: 로그만 기록 (Mock)

# [Exchange Setting]
EXCHANGE_ID = 'binance'  # binance, bybit, etc.

# [Risk Management Constants] - 요청하신 값 반영
INITIAL_CAPITAL = 10_000
PORTFOLIO_RISK_CAP = 0.01       # 1% (전체 리스크 한도)
PER_TRADE_RISK = 0.0025         # 0.25% (개별 거래 리스크)
MAX_LEVERAGE = 6
MAX_POSITIONS = 3
STOP_ATR = 3                    # 손절 거리 (ATR 배수)
ATR_PERIOD = 14
TRADING_FEE_RATE = 0.0005

# [Timeframe Setting]
SELECTION_TIMEFRAME = '1h'      # 종목 선정용 (1시간봉)
PA_TIMEFRAME = '15m'            # 매매 신호용 (15분봉)
LOOKBACK_WINDOW = 100           # 데이터 조회 개수
SELECTION_TOP_N = 3             # 시간당 선정할 후보군 수

# [Logging & Data Paths]
LOG_DIR = "./logs/mock_live"              # 로그 저장 폴더
SYSTEM_LOG_FILE = "system.log"  # 시스템 로그 파일명
TRADE_CSV_FILE = "trade_history.csv" # 거래 내역 파일명

client = Client(API_KEY, SECRET_KEY)
info = client.futures_exchange_info()

# ====================================================
# 2. 사용자 알고리즘 주입 (User Algorithms)
# ====================================================

# 실제 사용 시에는 파일에서 읽어오거나 여기에 코드를 붙여넣습니다.
# (함수 형태: calculate_ranking_score, generate_signals 등)

SELECTION_ALGO_CODE = """
def calculate_ranking_score(opens: pd.DataFrame, highs: pd.DataFrame, lows: pd.DataFrame, closes: pd.DataFrame, volumes: pd.DataFrame) -> pd.DataFrame:

    # --- Strategy Parameters (from Target Strategy Description) ---
    ATR_PERIOD = 14
    VOLUME_AVG_PERIOD = 20
    VOLUME_MULTIPLIER = 1.8  # Volume must be 1.8x its rolling average for conviction
    MOMENTUM_THRESHOLD = 1.5 # Body momentum must be > 1.5 * ATR
    EMA_PERIOD = 20          # Short-term trend filter and exit trigger

    # --- Step 1: Calculate Core Indicators ---
    # 1. True Range (TR) and Average True Range (ATR)
    prev_closes = closes.shift(1)
    
    tr_high_low = highs - lows
    tr_high_prev_close = (highs - prev_closes).abs()
    tr_low_prev_close = (lows - prev_closes).abs()

    # Element-wise maximum across the three TR components for all symbols
    # The original code incorrectly used DataFrame.max() with another DataFrame as argument,
    # which is not supported for element-wise comparison and caused the TypeError.
    # Use np.maximum for element-wise comparison across DataFrames.
    tr = np.maximum(tr_high_low, np.maximum(tr_high_prev_close, tr_low_prev_close))
    
    atr = tr.rolling(ATR_PERIOD).mean()
    # Replace any zero ATR values with a small number to prevent division by zero errors.
    # Fill leading NaNs for ATR calculation to ensure proper division later.
    atr_safe = atr.replace(0, np.nan).ffill().bfill().fillna(1e-9) 

    # 2. Rolling Mean Volume
    rolling_avg_volumes = volumes.rolling(VOLUME_AVG_PERIOD).mean()

    # 3. Exponential Moving Average (EMA) for trend filter
    ema = closes.ewm(span=EMA_PERIOD, adjust=False).mean()

    # --- Step 2: "Impulse Conviction Score" (ICS) ---
    # Measures the presence of strong, volume-validated impulse potential aligned with the EMA.
    
    # Body strength normalized by ATR
    body_strength_long = (closes - opens) / atr_safe
    body_strength_short = (opens - closes) / atr_safe
    
    # Volume ratio relative to its rolling average
    volume_ratio = volumes / rolling_avg_volumes

    # Calculate impulse potential: combination of body strength ratio and volume conviction ratio.
    # Clipping ensures that extreme outliers don't disproportionately skew the score,
    # while still rewarding values significantly above the thresholds. Max clip at 3x threshold.
    long_impulse_potential = (body_strength_long.clip(lower=0) / MOMENTUM_THRESHOLD).clip(upper=3) * \
                             (volume_ratio / VOLUME_MULTIPLIER).clip(upper=3)
    short_impulse_potential = (body_strength_short.clip(lower=0) / MOMENTUM_THRESHOLD).clip(upper=3) * \
                              (volume_ratio / VOLUME_MULTIPLIER).clip(upper=3)

    # Filter impulse potential by EMA alignment and candle direction
    long_aligned_impulse = long_impulse_potential * (closes > ema) * (closes > opens)
    short_aligned_impulse = short_impulse_potential * (closes < ema) * (opens > closes)

    # Average the combined aligned impulse potential over a recent period
    # This indicates how often an asset shows good, EMA-aligned impulse characteristics.
    ICS = (long_aligned_impulse + short_aligned_impulse).rolling(VOLUME_AVG_PERIOD).mean()

    # --- Step 3: "EMA Trend Durability Score" (ETS) ---
    # Measures how cleanly price maintains its trend relative to the EMA, avoiding whipsaws.
    # A robust environment for the strategy means trends persist without premature EMA crossovers.

    # 1. Determine EMA state (1: above, -1: below, 0: on/equal)
    ema_state = pd.DataFrame(np.sign(closes - ema), index=closes.index, columns=closes.columns)

    # 2. Identify changes in EMA state to group continuous trend periods
    # `state_change_id` increments each time the EMA state changes for a symbol.
    # The `groupby` operation expects a 1-dimensional Series as its `by` argument.
    # Since `ema_state` is a DataFrame with multiple symbol columns, `state_change_id`
    # calculated globally would also be a DataFrame, causing the "not 1-dimensional" error.
    # We need to apply this logic column-wise for each symbol.

    # Initialize an empty DataFrame to store the trend duration for each symbol
    trend_duration = pd.DataFrame(index=ema_state.index, columns=ema_state.columns, dtype=float)

    for column in ema_state.columns:
        # Get the Series for the current symbol's EMA state
        symbol_ema_state = ema_state[column]
        
        # Calculate the state change ID for this specific symbol (Series)
        # This `symbol_state_change_id` is a 1-dimensional Series.
        symbol_state_change_id = (symbol_ema_state.diff().fillna(0).abs() > 0).cumsum()
        
        # Group the `symbol_ema_state` Series by its `symbol_state_change_id` Series
        # and transform to get the size of each group (i.e., the duration of the trend).
        # The result is a Series, which is then assigned to the corresponding column in `trend_duration`.
        trend_duration[column] = symbol_ema_state.groupby(symbol_state_change_id).transform('size')
    
    # Normalize trend duration by EMA_PERIOD. Longer trends score higher.
    # A value > 1 means the trend duration is longer than the EMA_PERIOD, indicating strong persistence.
    normalized_trend_duration = trend_duration / EMA_PERIOD
    
    # 4. Measure the magnitude of price's divergence from EMA, normalized by ATR.
    # We want trends that not only last long but also move significantly away from the EMA.
    abs_ema_dist_norm = (closes - ema).abs() / atr_safe

    # Combine normalized trend duration with normalized EMA divergence magnitude.
    # Take a rolling average to reflect recent trend durability.
    ETS = (normalized_trend_duration * abs_ema_dist_norm).rolling(EMA_PERIOD).mean()

    # --- Step 4: Combine Scores for Final Ranking ---
    # A multiplicative combination ensures that an asset must exhibit both
    # strong impulse characteristics AND a clean, durable trend environment to score high.
    # If either component is low, the overall score will be low.
    ranking_score = ICS * ETS

    # Fill any remaining NaNs (e.g., from initial rolling periods) with 0,
    # indicating lowest priority for ranking.
    ranking_score = ranking_score.fillna(0)

    return ranking_score
"""

PA_ALGO_CODE = """
def generate_signals(opens, highs, lows, closes, volumes):
    # Strategy Parameters
    ATR_PERIOD = 14
    VOLUME_AVG_PERIOD = 20
    VOLUME_MULTIPLIER = 1.8  # Volume must be 1.8x its rolling average for conviction
    MOMENTUM_THRESHOLD = 1.5 # Body momentum (close-open or open-close) must be > 1.5 * ATR
    EMA_PERIOD = 20          # Short-term trend filter and exit trigger

    # --- Calculations ---
    # 1. True Range (TR) and Average True Range (ATR)
    prev_closes = closes.shift(1)
    
    # Calculate True Range components using numpy for vectorized operations
    high_minus_low = highs.values - lows.values
    high_minus_prev_close_abs = np.abs(highs.values - prev_closes.values)
    low_minus_prev_close_abs = np.abs(lows.values - prev_closes.values)

    # Stack the arrays and use np.maximum.reduce to get element-wise max for each symbol
    # Handle potential NaN values from prev_closes by propagating them
    tr_values = np.maximum.reduce([high_minus_low, high_minus_prev_close_abs, low_minus_prev_close_abs])
    tr = pd.DataFrame(tr_values, index=closes.index, columns=closes.columns)
    
    # Apply rolling mean for ATR
    atr = tr.rolling(ATR_PERIOD).mean()

    # 2. Rolling Mean Volume
    rolling_avg_volumes = volumes.rolling(VOLUME_AVG_PERIOD).mean()

    # 3. Exponential Moving Average (EMA) for trend filter
    ema = closes.ewm(span=EMA_PERIOD, adjust=False).mean()

    # 4. Body Momentum (normalized by ATR)
    # Replace any zero ATR values with a small number to prevent division by zero errors.
    # Fill leading NaNs for ATR calculation to ensure proper division later.
    atr_safe = atr.replace(0, np.nan).ffill().bfill().fillna(1e-9) 

    long_body_strength = (closes - opens) / atr_safe
    short_body_strength = (opens - closes) / atr_safe

    # --- Entry Conditions ---
    # Long Entry: Strong bullish candle (green), high relative volume, price above EMA
    long_entry_condition = (
        (closes > opens) & # Candle is bullish
        (long_body_strength > MOMENTUM_THRESHOLD) & # Body strength exceeds threshold
        (volumes > VOLUME_MULTIPLIER * rolling_avg_volumes) & # Volume confirmation
        (closes > ema) # Price is above EMA (uptrend confirmation)
    )

    # Short Entry: Strong bearish candle (red), high relative volume, price below EMA
    short_entry_condition = (
        (opens > closes) & # Candle is bearish
        (short_body_strength > MOMENTUM_THRESHOLD) & # Body strength exceeds threshold
        (volumes > VOLUME_MULTIPLIER * rolling_avg_volumes) & # Volume confirmation
        (closes < ema) # Price is below EMA (downtrend confirmation)
    )

    # Note: long_entry_condition and short_entry_condition are mutually exclusive
    # because (closes > ema) and (closes < ema) cannot both be true at the same time.

    # --- Exit Conditions (Reversion-based) ---
    # Long Exit: Close drops below EMA
    long_exit_condition = (closes < ema)
    # Short Exit: Close rises above EMA
    short_exit_condition = (closes > ema)

    # --- Signal Generation (Vectorized State Management) ---
    # Initialize a DataFrame to store the desired position state (1: Long, -1: Short, 0: Flat)
    # Using NaN initially allows ffill() to correctly propagate positions.
    desired_position_state = pd.DataFrame(np.nan, index=closes.index, columns=closes.columns)

    # Apply entry signals:
    # If a long entry condition is met, set desired_position_state for that bar to 1.
    desired_position_state[long_entry_condition] = 1
    # If a short entry condition is met, set desired_position_state for that bar to -1.
    desired_position_state[short_entry_condition] = -1

    # Calculate the previous effective position based on all decisions up to the prior bar.
    # .ffill() propagates the last determined state (1, -1, or NaN).
    # .shift(1) gives us the state from the *previous* bar's close.
    # .fillna(0) treats any initial NaNs (before any signals) as a flat position.
    prev_effective_position = desired_position_state.ffill().shift(1).fillna(0)

    # Apply exit signals based on previous state:
    # If we were long (prev_effective_position == 1) AND a long exit condition is met on the current bar,
    # then set desired_position_state for the current bar to 0 (flat).
    desired_position_state[long_exit_condition & (prev_effective_position == 1)] = 0
    # If we were short (prev_effective_position == -1) AND a short exit condition is met on the current bar,
    # then set desired_position_state for the current bar to 0 (flat).
    desired_position_state[short_exit_condition & (prev_effective_position == -1)] = 0

    # Final signals DataFrame:
    # .ffill() propagates the desired state (1, -1, or 0) forward to fill any remaining NaNs
    # (e.g., if an entry happened, it stays that state until an exit or another entry).
    # .fillna(0) handles any initial NaNs or periods where no signal was ever generated.
    # .astype(int) converts the final signals to integer type (-1, 0, 1).
    signals = desired_position_state.ffill().fillna(0).astype(int)
    
    # The output `signals` DataFrame represents the desired position (1: long, -1: short, 0: flat)
    # at the close of the current bar (t), which will be executed at the open of the next bar (t+1).
    return signals
"""

# ====================================================
# 3. 라이브 트레이더 클래스
# ====================================================

class LiveTradingSystem:
    def __init__(self, is_live=False):
        self.is_live = is_live
        self.equity = INITIAL_CAPITAL
        self.base_capital = INITIAL_CAPITAL
        
        self.open_positions = []        
        self.candidates = []            
        self.candidate_scores = {}      
        self.last_selection_hour = -1   
        
        # ---------------------------------------------------
        # [Logger Setup] 콘솔 출력 + 파일 저장 동시 수행
        # ---------------------------------------------------
        # ---------------------------------------------------
        # [Logger Setup] 중복 방지 로직 강화
        # ---------------------------------------------------
        if not os.path.exists(LOG_DIR):
            os.makedirs(LOG_DIR)
            
        self.logger = logging.getLogger("LiveTrader")
        
        # [수정] 기존에 등록된 핸들러가 있다면 모두 제거하여 중복 출력 방지
        if self.logger.hasHandlers():
            self.logger.handlers.clear()
            
        self.logger.setLevel(logging.INFO)
        # 전파 방지 (상위 로거로 로그가 전달되어 중복 출력되는 것 차단)
        self.logger.propagate = False 
        
        # 1. 파일 핸들러
        file_handler = logging.FileHandler(os.path.join(LOG_DIR, SYSTEM_LOG_FILE), encoding='utf-8')
        file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
        
        # 2. 스트림 핸들러 (콘솔 출력)
        stream_handler = logging.StreamHandler()
        stream_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
        
        # 핸들러 추가
        self.logger.addHandler(file_handler)
        self.logger.addHandler(stream_handler)

        # ---------------------------------------------------
        # 거래소 초기화
        # ---------------------------------------------------
        self.exchange = getattr(ccxt, EXCHANGE_ID)({
            'apiKey': API_KEY,
            'secret': SECRET_KEY,
            'enableRateLimit': True,
            'options': {'defaultType': 'future'}
        })
        
        # 알고리즘 로드
        self.selection_scope = {'pd': pd, 'np': np}
        self.pa_scope = {'pd': pd, 'np': np}
        exec(SELECTION_ALGO_CODE, self.selection_scope)
        exec(PA_ALGO_CODE, self.pa_scope)
        
        self.logger.info(f"🚀 System Initialized. Mode: {'REAL LIVE' if self.is_live else 'MOCK TRADING'}")
        self.logger.info(f"📁 Logs will be saved to: {LOG_DIR}")

    # ----------------------------------------------------------------
    # [Helper] CSV 거래 기록 저장 함수 (수정됨)
    # ----------------------------------------------------------------
    def save_trade_to_csv(self, event_type, data):
        """거래 발생 시 CSV 저장 (Entry/Exit 시간, Run Up/Down 추가)"""
        filepath = os.path.join(LOG_DIR, TRADE_CSV_FILE)
        file_exists = os.path.isfile(filepath)
        
        # [수정] 컬럼 확장
        fieldnames = [
            'timestamp', 'event', 'symbol', 'direction', 'price', 
            'size', 'notional', 'pnl', 'reason', 'equity', 'mode',
            'entry_time', 'exit_time', 'run_up', 'run_down' # <--- 추가된 칼럼
        ]
        
        try:
            with open(filepath, mode='a', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                
                if not file_exists:
                    writer.writeheader()
                
                # 데이터 매핑
                row = {
                    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                    'event': event_type,
                    'symbol': data.get('symbol'),
                    'direction': 'LONG' if data.get('direction') == 1 else 'SHORT',
                    'price': f"{data.get('price'):.4f}",
                    'size': f"{data.get('size'):.4f}",
                    'notional': f"{data.get('notional'):.2f}",
                    'pnl': f"{data.get('pnl', 0):.4f}",
                    'reason': data.get('reason', ''),
                    'equity': f"{self.equity:.2f}",
                    'mode': 'REAL' if self.is_live else 'MOCK',
                    
                    # [추가] 상세 시간 및 성과 지표
                    'entry_time': data.get('entry_time', ''),
                    'exit_time': data.get('exit_time', ''),
                    'run_up': f"{data.get('run_up', 0):.2f}%" if data.get('run_up') is not None else '',
                    'run_down': f"{data.get('run_down', 0):.2f}%" if data.get('run_down') is not None else ''
                }
                writer.writerow(row)
                
        except Exception as e:
            self.logger.error(f"❌ Failed to save trade to CSV: {e}")

    # ----------------------------------------------------------------
    # [Data] 데이터 수집
    # ----------------------------------------------------------------
    def fetch_market_data(self, symbols, timeframe='15m', limit=100):
        """바이낸스 선물 15분봉 데이터 전용 수집"""
        data = {'open': {}, 'high': {}, 'low': {}, 'close': {}, 'volume': {}}
        if not symbols: return None

        for sym in symbols:
            try:
                # 바이낸스 선물에서 15분봉 데이터 호출
                ohlcv = self.exchange.fetch_ohlcv(sym, timeframe, limit=limit)
                df = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
                df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
                df.set_index('timestamp', inplace=True)
                
                # 데이터가 비어있지 않은지 확인
                if df.empty: continue
                
                data['open'][sym] = df['open']
                data['high'][sym] = df['high']
                data['low'][sym] = df['low']
                data['close'][sym] = df['close']
                data['volume'][sym] = df['volume']
                
            except Exception as e:
                self.logger.error(f"❌ {sym} 15분봉 데이터 수집 실패: {e}")
                
        return {k: pd.DataFrame(v).ffill() for k, v in data.items()}

    def get_universe(self):
        info = client.futures_exchange_info()
        symbols = [
            s["symbol"]
            for s in info["symbols"]
            if s["symbol"].endswith("USDT") and s["status"] == 'TRADING'
        ]
        exclude = ["USDCUSDT", "USTCUSDT", "BUSDUSDT", "FDUSDUSDT"]

        return [s for s in symbols if s not in exclude]

    def calculate_atr(self, high, low, close, period=14):
        tr1 = high - low
        tr2 = (high - close.shift(1)).abs()
        tr3 = (low - close.shift(1)).abs()
        tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        return tr.rolling(period).mean().iloc[-1]

    # ----------------------------------------------------------------
    # [Step 1] Selection Logic
    # ----------------------------------------------------------------
    def run_hourly_selection(self):
        self.logger.info("🕐 Running Hourly Selection Algorithm...")
        universe = self.get_universe()
        market_data_1h = self.fetch_market_data(universe, SELECTION_TIMEFRAME)
        
        if not market_data_1h:
            self.logger.warning("❌ Selection Failed: No Data.")
            return

        calc_score_fn = self.selection_scope.get('calculate_ranking_score')
        
        try:
            scores = calc_score_fn(
                market_data_1h['open'], market_data_1h['high'], market_data_1h['low'],
                market_data_1h['close'], market_data_1h['volume']
            )
            latest_scores = scores.iloc[-1].sort_values(ascending=False)
            self.candidates = latest_scores.head(SELECTION_TOP_N).index.tolist()
            self.candidate_scores = latest_scores.to_dict()
            
            self.logger.info(f"✅ Candidates: {self.candidates}")
            
        except Exception as e:
            self.logger.error(f"❌ Selection Algo Error: {e}")
            traceback.print_exc()

    # ----------------------------------------------------------------
    # [Step 2] PA Logic
    # ----------------------------------------------------------------
    def run_pa_check_and_trade(self):
        if not self.candidates:
            self.logger.info("ℹ️ No candidates. Waiting...")
            return

        market_data_15m = self.fetch_market_data(self.candidates, PA_TIMEFRAME)
        if not market_data_15m: return

        gen_signal_fn = self.pa_scope.get('generate_signals')
        try:
            signals = gen_signal_fn(
                market_data_15m['open'], market_data_15m['high'], market_data_15m['low'],
                market_data_15m['close'], market_data_15m['volume']
            )
            
            self.check_exit_signals(market_data_15m, signals)
            self.process_entry_signals(market_data_15m, signals)
            
        except Exception as e:
            self.logger.error(f"❌ PA Algo Error: {e}")
            traceback.print_exc()

    # ----------------------------------------------------------------
    # [Step 3] Execution & Risk
    # ----------------------------------------------------------------
    def process_entry_signals(self, market_data, signals):
        last_signals = signals.iloc[-1]
        last_closes = market_data['close'].iloc[-1]
        entry_queue = []
        current_holdings = [p['symbol'] for p in self.open_positions]

        for sym in self.candidates:
            if sym in current_holdings: continue
            
            sig = last_signals.get(sym, 0)
            if sig == 0: continue
            
            prio = self.candidate_scores.get(sym, 0)
            entry_queue.append({
                'symbol': sym, 'direction': sig,
                'price': last_closes[sym], 'priority': prio
            })
            
        entry_queue.sort(key=lambda x: x['priority'], reverse=True)
        
        for trade in entry_queue:
            self.execute_trade_with_risk(trade, market_data)

    def execute_trade_with_risk(self, trade, market_data):
        if len(self.open_positions) >= MAX_POSITIONS:
            self.logger.info("🚫 Max Positions Reached.")
            return

        sym = trade['symbol']
        atr = self.calculate_atr(market_data['high'][sym], market_data['low'][sym], market_data['close'][sym])
        
        if np.isnan(atr) or atr == 0: return

        risk_amount = self.base_capital * PER_TRADE_RISK
        risk_per_share = atr * STOP_ATR
        position_size = risk_amount / risk_per_share
        notional_value = position_size * trade['price']
        
        current_pf_risk = sum(p['risk_amount'] for p in self.open_positions)
        if current_pf_risk + risk_amount > self.base_capital * PORTFOLIO_RISK_CAP:
            self.logger.warning(f"⚠️ Portfolio Risk Cap Reached ({sym}).")
            return
            
        current_pf_notional = sum(p['notional'] for p in self.open_positions)
        if current_pf_notional + notional_value > self.base_capital * MAX_LEVERAGE:
            self.logger.warning(f"⚠️ Max Leverage Exceeded ({sym}).")
            return
            
        stop_price = trade['price'] - (atr * STOP_ATR) if trade['direction'] == 1 else trade['price'] + (atr * STOP_ATR)
        
        self.record_entry(trade, position_size, notional_value, risk_amount, stop_price)

    # ----------------------------------------------------------------
    # [Logic] 포지션 추적 및 업데이트 (신규 추가)
    # ----------------------------------------------------------------
    def update_position_stats(self, market_data):
        """보유 포지션의 최고가/최저가(Run Up/Down 계산용) 실시간 업데이트"""
        for pos in self.open_positions:
            sym = pos['symbol']
            if sym in market_data['close'].columns:
                curr_price = market_data['close'][sym].iloc[-1]
                
                # 최고가/최저가 갱신
                pos['max_price'] = max(pos['max_price'], curr_price)
                pos['min_price'] = min(pos['min_price'], curr_price)

    # ----------------------------------------------------------------
    # [Step 3] 진입 기록 (수정됨)
    # ----------------------------------------------------------------
    def record_entry(self, trade, size, notional, risk_amt, stop_price):
        # 1. 실제/모의 주문 실행 (생략)
        if self.is_live: pass
        
        # 2. 로그 출력
        entry_time_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        self.logger.info(f"⚡ ENTRY: {trade['symbol']} | Price: {trade['price']} | Time: {entry_time_str}")

        # 3. CSV 저장 (ENTRY 시점에는 Run Up/Down 없음)
        self.save_trade_to_csv("ENTRY", {
            'symbol': trade['symbol'],
            'direction': trade['direction'],
            'price': trade['price'],
            'size': size,
            'notional': notional,
            'reason': 'Signal',
            'pnl': 0,
            'entry_time': entry_time_str, # Entry Time 기록
            'exit_time': '',
            'run_up': None,
            'run_down': None
        })

        # 4. 메모리 업데이트 (추적 변수 초기화)
        new_pos = {
            'symbol': trade['symbol'], 
            'direction': trade['direction'],
            'entry_price': trade['price'], 
            'position_size': size,
            'notional': notional, 
            'risk_amount': risk_amt,
            'stop_price': stop_price, 
            'entry_time': datetime.now(), # 객체로 저장
            
            # [추가] Run Up/Down 계산을 위한 추적 변수
            'max_price': trade['price'],
            'min_price': trade['price']
        }
        self.open_positions.append(new_pos)

    # ----------------------------------------------------------------
    # [Step 4] 청산 및 체크 (수정됨)
    # ----------------------------------------------------------------
    def check_exit_signals(self, market_data, signals):
        # [중요] 먼저 현재 가격으로 포지션 통계(High/Low) 업데이트
        self.update_position_stats(market_data)
        
        for pos in self.open_positions[:]:
            sym = pos['symbol']
            if sym not in market_data['close'].columns: continue
            
            curr_signal = signals.iloc[-1].get(sym, 0)
            curr_price = market_data['close'][sym].iloc[-1]
            
            # 청산 조건 확인
            algo_exit = (pos['direction'] == 1 and curr_signal <= 0) or \
                        (pos['direction'] == -1 and curr_signal >= 0)
            
            hit_sl = (pos['direction'] == 1 and curr_price <= pos['stop_price']) or \
                     (pos['direction'] == -1 and curr_price >= pos['stop_price'])
            
            if algo_exit or hit_sl:
                reason = "StopLoss" if hit_sl else "AlgoExit"
                self.close_position(pos, curr_price, reason)

    def close_position(self, pos, price, reason):
        # 1. 총 노출 금액(진입 시 + 청산 시) 계산
        entry_notional = pos['notional']                  # 진입 시 가치
        exit_notional = pos['position_size'] * price      # 청산 시 가치
        
        # 2. 수수료 계산 (진입 수수료 + 청산 수수료)
        entry_fee = entry_notional * TRADING_FEE_RATE
        exit_fee = exit_notional * TRADING_FEE_RATE
        total_fee = entry_fee + exit_fee
        
        # 3. 순수익(Net PnL) 계산: (매매차익 - 총 수수료)
        gross_pnl = (price - pos['entry_price']) * pos['position_size'] * pos['direction']
        net_pnl = gross_pnl - total_fee
        
        # 4. 자산 업데이트
        self.equity += net_pnl
        
        # 2. Run Up / Run Down 계산 (수익률 %)
        # Long: (High - Entry)/Entry, (Low - Entry)/Entry
        # Short: (Entry - Low)/Entry, (Entry - High)/Entry  <- 숏은 가격이 내려야 이득
        
        if pos['direction'] == 1: # Long
            run_up = (pos['max_price'] - pos['entry_price']) / pos['entry_price'] * 100
            run_down = (pos['min_price'] - pos['entry_price']) / pos['entry_price'] * 100
        else: # Short
            run_up = (pos['entry_price'] - pos['min_price']) / pos['entry_price'] * 100 # 숏은 최저가가 최대 수익
            run_down = (pos['entry_price'] - pos['max_price']) / pos['entry_price'] * 100 # 숏은 최고가가 최대 손실
            
        entry_time_str = pos['entry_time'].strftime('%Y-%m-%d %H:%M:%S')
        exit_time_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        # 로그 출력 시 수수료 정보 포함
        self.logger.info(f"🏁 EXIT: {pos['symbol']} | Net PnL: {net_pnl:.2f} (Fee: {total_fee:.2f}) | Up: {run_up:.2f}% | Down: {run_down:.2f}%")
        
        # 6. CSV 저장 (pnl 자리에 net_pnl 저장)
        self.save_trade_to_csv("EXIT", {
            'symbol': pos['symbol'],
            'direction': pos['direction'],
            'price': price,
            'size': pos['position_size'],
            'notional': exit_notional,
            'reason': reason,
            'pnl': net_pnl, # 수수료가 차감된 순수익 저장
            'entry_time': entry_time_str,
            'exit_time': exit_time_str,
            'run_up': run_up,
            'run_down': run_down
        })

        self.open_positions.remove(pos)

    # ----------------------------------------------------------------
    # [Main Loop]
    # ----------------------------------------------------------------
    def run(self):
        self.logger.info("🟢 Starting Live Trading Loop...")
        
        while True:
            try:
                now = datetime.now()
                current_hour = now.hour
                
                if current_hour != self.last_selection_hour:
                    self.logger.info(f"\n🔔 New Hour ({current_hour}:00). Updating Candidates...")
                    self.run_hourly_selection()
                    self.last_selection_hour = current_hour
                
                self.run_pa_check_and_trade()
                
                self.logger.info(f"💤 Cycle Done. Waiting 60s... (Cand: {self.candidates}, Pos: {len(self.open_positions)})")
                time.sleep(60)
                
            except KeyboardInterrupt:
                self.logger.info("🛑 Stopped by User.")
                break
            except Exception as e:
                self.logger.error(f"❌ Critical Error: {e}")
                traceback.print_exc()
                time.sleep(10)

if __name__ == "__main__":
    bot = LiveTradingSystem(is_live=IS_LIVE_TRADING)
    bot.run()


In [ ]:
from binance.client import Client
client = Client(API_KEY, SECRET_KEY)
info = client.futures_exchange_info()


In [12]:
symbols = [
            s["symbol"]
            for s in info["symbols"]
            if s["symbol"].endswith("USDT") and s["status"] == 'TRADING'
        ]

len(symbols)

539